In [32]:
import requests
import pandas as pd
import numpy as np
API_KEY = "765cdcab5c6c40b929e570ef81530f68412b4ba7"

def decode_age(code):
    if code == "0000":
        return "total"
    elif code == "0001":
        return "0"      # under 1 year
    elif code == "8599":
        return "85+"   # 85 and over
    else:
        return str(int(code) // 100)

In [33]:
url = "https://api.census.gov/data/2023/pep/charv"
params = {
    "get": "NAME,AGE,SEX,POP",
    "for": "state:*",
    "YEAR": "2023",
    "SEX": "0",
    "key": API_KEY
}
r = requests.get(url, params=params)
data = r.json()
df = pd.DataFrame(data[1:], columns=data[0])

df = df.loc[df.AGE.isin(['0001', 
                        '0100', '0200', '0300', '0400', '0500', 
                        '0600', '0700', '0800', '0900', '1000', 
                        '1100', '1200', '1300', '1400', '1500',
                        '1600', '1700', '1800', '1900', '2000',
                        '2100', '2200', '2300', '2400', '2500',
                        '2600', '2700', '2800', '2900', '3000',
                        '3100', '3200', '3300', '3400', '3500', 
                        '3600', '3700', '3800', '3900', '4000',
                        '4100', '4200', '4300', '4400', '4500',
                        '4600', '4700', '4800', '4900', '5000', 
                        '5100', '5200', '5300', '5400', '5500',
                        '5600', '5700', '5800', '5900', '6000', 
                        '6100', '6200', '6300', '6400', '6500',
                        '6600', '6700', '6800', '6900', '7000',
                        '7100', '7200', '7300', '7400', '7500',
                        '7600', '7700', '7800', '7900', '8000',
                        '8100', '8200', '8300', '8400', '8599'])].reset_index(drop=True)

# decode age
df["group_name"] = df.AGE.apply(decode_age)

# keep selected columns
df = df[["NAME", "AGE", "group_name", "POP"]]
df.rename(columns={"NAME": "location", "POP": "value"}, inplace=True)
df["value"] = df["value"].astype(int)

# fix 84 and 85+ to match epydemix format
df.loc[df["group_name"] == '84', "group_name"] = '84+'
df.loc[df["group_name"] == '85+', "group_name"] = '84+'

# group by state and age
df = df.groupby(["location", "group_name"], as_index=False).sum()
df["location"] = df["location"].str.replace(" ", "_")

# sort 
ages = np.concatenate((np.arange(len(df_example.group_name)-1), ["84+"]))
df['group_name'] = pd.Categorical(df['group_name'], categories=ages, ordered=True)
df = df.sort_values('group_name').reset_index(drop=True)
df.head()

,location,group_name,AGE,value
0,Alabama,0,0001,57885
1,New_Hampshire,0,0001,12164
2,Connecticut,0,0001,34935
3,New_Jersey,0,0001,102477
4,New_Mexico,0,0001,20589


In [34]:
locations = pd.read_csv("./locations.csv")
locations = locations.loc[locations.location.str.startswith("United_States")]
locations = locations.loc[locations.location != "United_States"]
locations["location"] = locations["location"].str.replace("United_States_", "")
us_locations_epydemix = locations.location.unique()

us_locations_epydemix

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District_of_Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New_Hampshire',
       'New_Jersey', 'New_Mexico', 'New_York', 'North_Carolina',
       'North_Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode_Island', 'South_Carolina', 'South_Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West_Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [35]:
for location in us_locations_epydemix:
    df_location = df.loc[df.location == location]
    df_location[["group_name", "value"]].to_csv(f"./data/United_States_{location}/demographic/age_distribution.csv", index=False)

In [ ]:
# Create US total 
df[["group_name", "value"]].groupby("group_name", as_index=False).sum().to_csv("./data/United_States/demographic/age_distribution.csv", index=False)

In [46]:
# fix sources
locations = pd.read_csv("./locations.csv")

locations.loc[locations.population_source == "https://population.un.org/wpp/Download/Standard/CSV/", "population_source"] = "https://population.un.org/wpp/"
locations.loc[locations.location.str.startswith("United_States"), "population_source"] = "https://api.census.gov/data/2023/pep/charv"
locations.head()

,location,primary_contact_source,mistry_2021,prem_2021,prem_2017,population_source
0,Afghanistan,prem_2021,False,True,False,https://population.un.org/wpp/
1,Albania,prem_2021,False,True,True,https://population.un.org/wpp/
2,Algeria,prem_2021,False,True,True,https://population.un.org/wpp/
3,Andorra,prem_2017,False,False,True,https://population.un.org/wpp/
4,Angola,prem_2021,False,True,False,https://population.un.org/wpp/


In [48]:
locations.to_csv("./locations.csv", index=False)